# V13: ingredient lists and recipe selection

27 November 2026. The user reported that a request for other egg recipes was refused, a list request was misclassified as injection, a carrot search returned only one match, and a Bibimbap request was refused. The v12 replay reproduced all four problems: `results/user_egg_carrot_v1_app_ollama_conversation_v12_20261127_101242_652900.json`.

## Design choices

`chef/recipe_catalog_v1.py` adds a structured lookup for a limited set of full ingredient-list requests. It searches ingredient names in **all** stored V4 recipe records instead of asking Qwen to infer an exhaustive list from the top retrieved fragments. Singular/plural word matching distinguishes `egg` from words such as `eggplant`; tomato includes tomato paste. This is a catalogue lookup, not vector search. Each matching recipe's stored ingredient record is included in the source trace. Lists mean ingredients explicitly recorded by name; they are not allergen guarantees and may not include implicit ingredients.

To prevent the two false refusals, this narrow path accepts only complete recognized request formats and known ingredient names. It checks for explicit safety terms and recent allergy context; it does not call the model safety classifier for these constrained lookups. All other requests continue to the previous safety and LangGraph paths. This trades broad language coverage for predictable results: alternative wording and implicit dietary constraints may still be mishandled. Lists are not safety-certified or a substitute for checking labels.

Named selections such as `Bibimbap` or `give me the recipe for bibimbap` use structured source data. The Bibimbap preparation is a fixed adaptation of the stored recipe: the original recommends a runny yolk; the adaptation cooks the eggs until firm, adds a temperature/rest check for beef, and cites [FDA food-handling guidance](https://www.fda.gov/food/buy-store-serve-safe-food/safe-food-handling). The source and debug trace still retain the original, **unreviewed** source steps. Previously reviewed Menemen, Ratatouille, Borscht, French onion soup and Spanish tortilla preparations remain available. Other named recipes currently return ingredients with an explicit statement that reviewed steps are unavailable, rather than claiming that all preparation instructions are verified.

The new entry point is `app_ollama_conversation_v13.py`; earlier versions remain intact.

## Tests and evidence

* `results/catalogue_v1_20261127_101701_575200.json`: 14 live UI turns across the reported conversation, different ingredient and wording, an allergy follow-up, injection and hazardous input. All targeted assertions passed. The egg list matched 12 stored records and the carrot list 6. The alternative-egg list excluded the previously suggested Menemen. Each listed name had its own stored ingredient record in the trace.
* `results/safety_final_v4_20261127_101708_893100.json`: 11 selected safety and ordinary-use turns passed.
* `results/realistic_assessment_v1_20261127_101749_444900.json`: 21/21 targeted checks across seven earlier conversations passed.
* `results/tomato_egg_chat_v1_app_ollama_conversation_v13_20261127_101732_456700.json`: the previously reported hazard-to-eggs scenario was replayed, including the hazardous request remaining blocked and the later egg recipe returning.

These tests are finite examples, not proof that every question or safety risk is handled correctly. Ragas was not rerun for v13. The original undercooked source instructions remain visible in raw/source evidence for provenance and should not be treated as verified advice.

![Egg recipe list in the browser](results/screenshots_catalogue_v1/egg_recipes_v13.jpg)

![Bibimbap answer in the browser](results/screenshots_catalogue_v1/bibimbap_v13.jpg)

## Reproduce

With Ollama and `qwen2.5:7b` available, run from the project folder:

```powershell
.\.venv\Scripts\python.exe -m streamlit run v4\app_ollama_conversation_v13.py
.\.venv\Scripts\python.exe v4\test_catalogue_v1.py
```

Stop an old Streamlit process before starting the new version; an old tab can still be serving v12.


In [ ]:
from pathlib import Path
import json
root=Path.cwd()
if not (root/'results').exists(): root=root/'v4'
rows=json.loads((root/'results/catalogue_v1_20261127_101701_575200.json').read_text(encoding='utf-8'))
[(r['question'], r['route'], r['problems']) for r in rows]